<a href="https://colab.research.google.com/github/pstansfeld/MemPrO/blob/main/MemPrO_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## MemPrO on Google Colab

[MemPrO](https://doi.org/10.1021/acs.jctc.5c01433) predicts the orientation of a membrane protein in a lipid bilayer (single or double membrane), and [MemPrOD](https://github.com/pstansfeld/MemPrOD) predicts the membrane deformations around it.

This notebook installs MemPrO from PyPI, orients a structure from the PDB, AlphaFold DB or your own upload, shows the result, and ends with a short test summary.

* **Runtime:** a CPU runtime works (the free runtime has only two cores, so a typical protein takes a few minutes). On a GPU runtime MemPrO uses the GPU automatically; set `Device` to `cpu` to avoid it.
* **Steps:** run the cells in order (*Runtime → Run all*). Settings are in the first cell.

In [ ]:
#@title Settings
Source = "RCSB PDB" #@param ["RCSB PDB", "AlphaFold DB", "Upload"]
#@markdown PDB code (RCSB) or UniProt accession (AlphaFold DB); ignored for uploads.
Structure_ID = "4G1U" #@param {type:"string"}
#@markdown Number of starting orientations and minimisation iterations (MemPrO's `-ng` and `-ni`).
Grid_size = 16 #@param {type:"integer"}
Iterations = 150 #@param {type:"integer"}
#@markdown `auto` uses a GPU if the runtime has one.
Device = "auto" #@param ["auto", "cpu", "gpu"]
#@markdown For periplasm-spanning proteins (two membranes).
Dual_membrane = False #@param {type:"boolean"}
#@markdown Also predict membrane deformations with MemPrOD (single membrane only).
Run_MemPrOD = False #@param {type:"boolean"}

In [ ]:
#@title Install MemPrO
import importlib.metadata, subprocess, sys

def installed_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None

# Recorded before installing: replacing the runtime's own JAX is what used to break MemPrO on Colab
jax_before = installed_version("jax")
packages = ["mempro", "py3Dmol"] + (["memprod"] if Run_MemPrOD and not Dual_membrane else [])
result = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, capture_output=True, text=True)
print(result.stdout[-2000:], result.stderr[-2000:])
jax_after = installed_version("jax")
print(f"jax before: {jax_before} | after: {jax_after} | mempro {installed_version('mempro')} | memprod {installed_version('memprod')}")
if jax_before and jax_after != jax_before:
    print("NOTE: pip changed the runtime's JAX version. If anything fails below, use Runtime > Restart session and run again.")

# The mempro / MemPrOD commands are installed next to this Python, which is not always on PATH
import os, shutil
def command_path(name):
    local = os.path.join(os.path.dirname(sys.executable), name)
    return local if os.path.exists(local) else (shutil.which(name) or name)

In [ ]:
#@title Check the environment
import os, platform, shutil
print("Python", platform.python_version(), "|", os.cpu_count(), "CPU cores")
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
# JAX is checked in a separate process so this notebook never imports it (MemPrO configures it at start-up)
check = subprocess.run([sys.executable, "-c", "import jax; print('jax', jax.__version__, '| devices:', jax.devices())"], capture_output=True, text=True)
print(check.stdout.strip() or check.stderr[-1000:])

In [ ]:
#@title Get the structure
import json, urllib.request

os.makedirs("/content/mempro_run" if os.path.isdir("/content") else "mempro_run", exist_ok=True)
work_dir = os.path.abspath("/content/mempro_run" if os.path.isdir("/content") else "mempro_run")
os.chdir(work_dir)

if Source == "Upload":
    from google.colab import files
    uploaded = files.upload()
    pdb_file = next(iter(uploaded))
elif Source == "RCSB PDB":
    pdb_file = f"{Structure_ID.strip().upper()}.pdb"
    urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_file}", pdb_file)
else:
    accession = Structure_ID.strip().upper()
    meta = json.load(urllib.request.urlopen(f"https://alphafold.ebi.ac.uk/api/prediction/{accession}"))[0]
    pdb_file = f"AF-{accession}.pdb"
    urllib.request.urlretrieve(meta["pdbUrl"], pdb_file)

n_atoms = sum(1 for line in open(pdb_file) if line.startswith("ATOM"))
print(f"{pdb_file}: {n_atoms} atoms")

In [ ]:
#@title Run MemPrO
import time

command = [command_path("mempro"), "-f", pdb_file, "-ng", str(Grid_size), "-ni", str(Iterations), "-dev", Device, "-o", "Orient/"]
if Dual_membrane:
    command.append("-dm")
print("mempro", " ".join(command[1:]))
start = time.time()
mempro_run = subprocess.run(command, capture_output=True, text=True)
mempro_time = time.time() - start
print(mempro_run.stdout[-3000:])
if mempro_run.returncode != 0:
    print(mempro_run.stderr[-3000:])
print(f"MemPrO finished with exit code {mempro_run.returncode} in {mempro_time:.0f} s")

In [ ]:
#@title Results
from IPython.display import Image, display
import py3Dmol

print(open("Orient/orientation.txt").read())
for plot in ["Orient/local_minima_orientation.png", "Orient/Rank_1/Z_potential_curve.png"]:
    if os.path.exists(plot):
        display(Image(plot, width=420))

view = py3Dmol.view(width=700, height=500)
view.addModel(open("Orient/Rank_1/oriented_rank_1.pdb").read(), "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.setStyle({"resn": "DUM"}, {"sphere": {"radius": 0.6, "color": "lightgrey"}})
view.zoomTo()
view.show()

In [ ]:
#@title Predict membrane deformations with MemPrOD (optional)
memprod_run = None
if Run_MemPrOD and not Dual_membrane:
    # MemPrOD needs the oriented protein only, without MemPrO's dummy membrane beads
    with open("Orient/Rank_1/oriented_rank_1.pdb") as src, open("protein_rank_1.pdb", "w") as dst:
        dst.writelines(line for line in src if " DUM " not in line)
    start = time.time()
    memprod_run = subprocess.run([command_path("MemPrOD"), "-f", "protein_rank_1.pdb", "-o", "Deformations/"], capture_output=True, text=True)
    memprod_time = time.time() - start
    print(memprod_run.stdout[-2000:])
    if memprod_run.returncode != 0:
        print(memprod_run.stderr[-3000:])
    print(f"MemPrOD finished with exit code {memprod_run.returncode} in {memprod_time:.0f} s")
    from IPython.display import SVG
    for plot in ["Deformations/Deformations_Upper.svg", "Deformations/Deformations_Lower.svg"]:
        if os.path.exists(plot):
            display(SVG(plot))
else:
    print("Skipped (set Run_MemPrOD in Settings; single membrane only)")

In [ ]:
#@title Test summary
def rank_1():
    rows = [line.split() for line in open("Orient/orientation.txt") if line.strip() and line.split()[0] == "1"]
    return (float(rows[0][1]), float(rows[0][2])) if rows else (None, None)

potential, hits = rank_1() if os.path.exists("Orient/orientation.txt") else (None, None)
checks = [
    ("pip kept the runtime's JAX", jax_before is None or jax_after == jax_before, f"{jax_before} -> {jax_after}"),
    ("MemPrO ran", mempro_run.returncode == 0, f"exit {mempro_run.returncode}, {mempro_time:.0f} s"),
    ("orientation.txt and rank 1 written", potential is not None and os.path.exists("Orient/Rank_1/oriented_rank_1.pdb"), f"rank 1 potential {potential}, {hits}% of starts"),
]
if memprod_run is not None:
    checks.append(("MemPrOD ran", memprod_run.returncode == 0 and os.path.exists("Deformations/Deformations.pdb"), f"exit {memprod_run.returncode}, {memprod_time:.0f} s"))
for name, ok, detail in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name:36s} {detail}")
print("\nAll checks passed" if all(ok for _, ok, _ in checks) else "\nSome checks failed: see the output above")

In [ ]:
#@title Download the results
# The archive is written next to the results folder, not inside it (it would otherwise try to include itself)
archive = shutil.make_archive(os.path.join(os.path.dirname(work_dir), "mempro_results"), "zip", work_dir)
print(f"{archive}: {os.path.getsize(archive)/1e6:.1f} MB")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass